### Non-additive PGS Demo Notebook - Neural Network

This notebook provides a minimal, standalone demonstration of how the neural network model was trained and evaluated for a single simulated phenotype in the paper Benchmarking non-additive genetic effects on polygenic prediction and machine learning-based approaches. It loads a demo dataset (one of the smaller synthetic phenotypes used in the manuscript), performs cross-validation using the same code structure as the full Snellius pipeline, runs Optuna-based hyperparameter tuning, and compares results to the published model outputs.

Phenotype:
- 100 causal SNPs
- total SNP heritability = 50%
- all causal SNPs have a dominance deviation ratio of k = -0.5

The total run time of the notebook (including loading + cleaning data and running the model) was ~ 4.3 hours using a 2021 MacBook Pro (Sonoma 14.5; M1 Max; 32GB RAM). 

NOTE: If you want to visualize the model training (e.g., train and tune loss function, variance explained) you can do so with Tensorboard using the SummaryWriter packaged (loaded below). Simply set the writer directory path in cell 4 and uncomment the lines in the model cell (will be noted). Launch tensorboard via the command line with (edit the path to logs to your own):      

```export LOGDIR=/path/to/logs```                      
```tensorboard --logdir "$LOGDIR"```                       

#### DATA & CODE
GitHub + Readme: https://github.com/nybell/non-add-paper/tree/main         
Zenodo repo: https://zenodo.org/records/17552313                                                 
Manuscript DOI: https://doi.org/10.1101/2025.10.10.25337750                            
Questions: n.y.bell@vu.nl                       

______________________________________________________________________________

### Load required packages

The cell below loads all required packages used in the demo. Information for setting up the Python environment can be found at the GitHub link above. 

The custom classes and functions ```DNN```, ```DNNObjective```, ```adjusted_r2```, ```get_architecture_params```, and ```split_data``` are imported from the ```model_definition.py```, which must be available in the same directory as the notebook (or on the Python path).

In [1]:
# import libraries
import os
import sys
import copy
import time
import torch
import optuna
import pickle
import random
import numpy as np
import pandas as pd
from tqdm import tqdm
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import r2_score
from torch.optim.lr_scheduler import ReduceLROnPlateau
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from torch.utils.data import DataLoader, TensorDataset
from optuna.samplers import GridSampler
from torch.utils.tensorboard import SummaryWriter
from model_definition import DNN, DNNObjective, adjusted_r2, get_architecture_params, split_data

/opt/miniconda3/envs/ml_models/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# create dataset class
class dataset():
    def __init__(self, x, y, ids):
        self.x = torch.tensor(x, device=device, dtype=torch.float32)        # MPSFloatType  torch.float32
        self.y = torch.tensor(y, device=device, dtype=torch.float32)        # .float32
        self.ids = ids
        self.length = self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx], self.ids[idx]

    def __len__(self):
        return self.length

In [3]:
# Set your desired seed for reproducibility
seed = 42
# Python's built-in random module
random.seed(seed)
# NumPy
np.random.seed(seed)
# PyTorch
torch.manual_seed(seed)

### Load data & prep for model

The cell below specifies the file path for:
- the input data file containing the simulated phenotypes + causal SNPs
- the results file for the same phenotype as published (given here for comparison)

These file paths should be editted to match your own copies of the data. 

Note: If you want to run the code for any of the other simulated phenotypes, they can be found on the Zenodo repository (https://zenodo.org/records/17552313) in the ```sim_data_ml_ready_071125.tar.gz``` file. All model results can be found on the GitHub at ```non-add-paper/results/model_out/```

In [ ]:
# set file path to data file
datafile = "/Users/nyb/demo_data/DATA_eur_nsnps100_h0.5_a0_d0.5_50k.txt"         # EDIT TO YOUR OWN FILE PATH

# set file path to published equivalent for same phenotype
pubfile = "/Users/nyb/demo_data/DNN_nsnps100_h0.5_a0_d0.5_50k_epochs100.pkl"     # EDIT TO YOUR OWN FILE PATH

# set path to tensorboard writer (if using)
# writer_path = "/Users/nyb/demo_data/tensorboard/"                                # EDIT TO YOUR OWN FILE PATH

The cell below loads the data and performs some initial cleaning:
- remove ```IID``` from the beginning of IID numbers 
- ```IID``` here numbers are just fillers for the PLINK .fam file, since this is a simulated data set using fake individuals
- fill NAs with ```0.0```
- replace ```"test"``` in ```split1``` with ```10```
- this is done to avoid errors later (split1 is not used for data splitting)
- ```data.head()``` checks data for (my and your) sanity

In [5]:
# load all data set
data = pd.read_csv(datafile, sep = "\t")
# remove "syn" from ids
data['IID'] = data['IID'].str.replace('syn', '').astype(int)
# fill NAs with 0.0
data = data.fillna(0.0)
# replace "test" with "10" in split1 column and convert to int
data['split1'] = data['split1'].replace("test", "10").astype(int)
# check data
data.head()

,FID,IID,father,mother,sex,phenotype,split1,split2,additive.prs,domdev.add.comp,...,chr5:93258428:G:A_A,chr16:66071133:A:G_G,chr3:183188020:A:G_A,chr1:194841459:A:G_G,chr8:24862989:G:A_G,chr8:4779193:T:C_C,chr10:83170668:G:T_T,chr4:146028406:A:G_G,chr12:4856240:A:C_A,chr4:14365883:A:G_A
0,syn1,1,0,0,0,5.063255,1,5,5.316734,0,...,1,0,0,0,1,1,2,1,0,0
1,syn10,10,0,0,0,5.162452,2,4,6.075860,0,...,1,0,0,2,0,1,1,0,1,0
2,syn100,100,0,0,0,4.745832,3,2,6.302860,0,...,2,0,0,0,0,0,1,0,1,1
3,syn1000,1000,0,0,0,5.045312,2,5,5.456551,0,...,0,0,0,1,1,0,1,0,1,1
4,syn10000,10000,0,0,0,5.626505,4,1,6.434155,0,...,0,1,0,0,0,1,0,0,0,1


The cell below drops columns that are not used as predictors in the NN model:
- ```FID```, ```father```, ```mother```, ```sex``` and polygenic scores ```additve.prs```, ```domdev.add.comp```, ```domdev.dom.comp```,
               ```domdev.prs```
- ```IID``` is dropped later in the model runs (in the ```split_data``` function)
- a ```try/except``` block is used in case ```domdev.add.comp``` is missing (wasn't always added in when formatting final data sets)

In [6]:
# drop columns that are not needed
try:
    # Try to drop all the specified columns
    data.drop(['FID', 'father', 'mother', 'sex',
               'additive.prs', 'domdev.add.comp', 'domdev.dom.comp',
               'domdev.prs'], axis=1, inplace=True)
except KeyError:
    # If there's an error, drop the same columns excluding 'domdev.add.comp'
    data.drop(['FID', 'father', 'mother', 'sex',
               'additive.prs', 'domdev.dom.comp',
               'domdev.prs'], axis=1, inplace=True)

Cell below performs one-hote encoding (essential for NN models to learn non-additive SNP effects):

- Separate non-SNP columns (`IID`, `split1`, `split2`, `phenotype`) from the SNP genotype matrix
- Apply OneHotEncoder to convert each SNP into binary indicator variables
- Convert the encoded output to a DataFrame and merge it back with the non-SNP columns
- Print the final data shape to confirm successful encoding and reconstruction

In [7]:
# Step 1: Separate the first three columns (IID, split1, phenotype) and SNP data
non_snp_columns = data[['IID', 'split1', 'split2', 'phenotype']]  # First three columns
snp_data = data.drop(columns=['IID', 'split1', 'split2', 'phenotype'])  # SNP data for encoding

# Step 2: One-hot encode SNPs
# Initialize the OneHotEncoder
encoder = OneHotEncoder(sparse_output=False, categories='auto')

# Fit and transform the SNP data
one_hot_encoded = encoder.fit_transform(snp_data)

# Convert the one-hot encoded data to a DataFrame (to facilitate merging)
one_hot_snp_data = pd.DataFrame(one_hot_encoded, index=data.index)

# Step 3: Merge the non-SNP columns back with the one-hot encoded SNP columns
# Concatenating the non-SNP columns (IID, split1, phenotype) with the one-hot encoded SNP data
data = pd.concat([non_snp_columns, one_hot_snp_data], axis=1)

# Verify final shape
print(" > Final data shape:", data.shape)

 > Final data shape: (50000, 297)


Cell below sets the GPU/CPU device. Will detect if one is available, and if not will set to CPU.
- MPS = Mac silison
- CUDA - NVIDIA GPU

In [8]:
# set device
print("| ... Setting device ... |")
if torch.cuda.is_available():
    device = torch.device("cuda")
    print(" > Using CUDA device:", device, " ... \n")
elif torch.has_mps == True:
    device = torch.device("mps")
    print(" > Using MPS device:", device, " ... \n")
else:
    device = torch.device("cpu")
    print(" > CUDA/MPS device not available. Using CPU:", device, " ... \n")
#     raise RuntimeError(" !! CUDA device not available. Aborting to avoid running on CPU !!")


| ... Setting device ... |
 > Using MPS device: mps  ... 



### Define hyperparameters & settings

This cell will:
- Define core training settings: number of epochs, Optuna trials, and batch size.
- Specify hyperparameter search space: learning rates, dropout probabilities, and L2 regularization strengths.
- Toggle architectural options such as batch normalization.
- Set the hyperparameter search strategy (e.g., "grid").

In [9]:
# Set hyperparameters & settings
epochs = 100
optuna_trials = 50
batch_size = 32
learning_rates = [1e-5, 1e-4]
dropout_probs = [0.0, 0.25, 0.5]
l2_values = [1e-6, 1e-5, 1e-4]
use_batch_norm = False
search_type = "grid"

### Run model 

Run the neural network across cross-validation splits.
For each split ((n = 5) using `split2` to split):
- Loop over 5 predefined cross-validation splits and generate train/validation/test sets
- Print dataset shapes and derive model input dimensionality
- Define network architecture parameters based on input size
- Wrap datasets in PyTorch Dataset and DataLoader objects.
- Create an Optuna objective function and run either grid search or random search to find the best learning rate, dropout probability, and L2 penalty.
- Instantiate the DNN with the selected hyperparameters and set up loss, optimizer, LR scheduler, and device placement
- Train the model for up to the set amount of (100) epochs with early stopping based on validation loss
- Restore the best-performing model and evaluate it on the test set.
- Compute performance metrics (R²) and record predictions, labels, and IDs for each split
- Store results in a dictionary and repeat for all CV splits

NOTE: If using Tensorboard, uncomment the relevant lines in the cell below (they are noted)

In [10]:
# --- time the entire cell ---
cell_start = time.time()

# save test data
test_data = {}

# start for loop
for split in range(1,6):

    # split data
    X_train, y_train, X_valid, y_valid, X_test, y_test, \
        ids_train, ids_tune, ids_test, tune_split = split_data(data, split_col = "split2",
                                                               split_num = split,
                                                               drop_col = "split1",
                                                               phenotype_col = "phenotype", 
                                                               fold_type = "cv")
    # print current split
    print("| ... CV split", tune_split, " ... |")
    print(" > Train set shape:", X_train.shape, y_train.shape)
    print(" > Validation set shape:", X_valid.shape, y_valid.shape)
    print(" > Test set shape:", X_test.shape, y_test.shape)

    # Define input and output dimensions
    input_dim = X_train.shape[1]
    output_dim = 1
    print(" > Input dimension:", input_dim)

    # define architecture
    hidden_dims, use_dropout, use_batch_norm = get_architecture_params(input_dim, batch_norm = use_batch_norm)
    print(" > Hidden layers:", hidden_dims)
    print(" > Use dropout:", use_dropout)
    print(" > Use batch norm:", use_batch_norm)
    print("")

    # ready training set
    trainset = dataset(X_train,y_train, ids_train)
    trainloader = DataLoader(trainset,batch_size=batch_size,shuffle=True)

    # ready tuning set
    validset = dataset(X_valid, y_valid, ids_tune)
    validloader = DataLoader(validset, batch_size=batch_size, shuffle=True)

    # ready test set
    testset = dataset(X_test, y_test, ids_test)
    testloader = DataLoader(testset, batch_size=batch_size, shuffle=True)

    # Create Optuna objective instance
    objective = DNNObjective(
        trainloader=trainloader,
        validloader=validloader,
        input_dim=input_dim,
        output_dim=output_dim, 
        epochs=epochs,
        hidden_dims=hidden_dims, 
        use_dropout=use_dropout, 
        learning_rate=learning_rates, 
        dropout_p=dropout_probs, 
        l2 = l2_values, 
        search_type=search_type
    )

    if search_type == "grid":
        print(" > Performing grid search ...")
        # define search space for learning rate and dropout probability
        search_space = {
            "learning_rate": learning_rates,
            "dropout_p": dropout_probs,
            "l2": l2_values,
        }
        sampler = GridSampler(search_space)
        study = optuna.create_study(direction="maximize", sampler=sampler)
        n_trials = len(learning_rates) * len(dropout_probs) * len(l2_values)
        study.optimize(objective, n_trials=n_trials)
    else:
        print(" > Performing random search ...")
        # Optimize using Optuna
        study = optuna.create_study(direction="maximize")
        study.optimize(objective, n_trials=optuna_trials)

    # Best hyperparameters
    print("Best hyperparameters:", study.best_params)
    print("Best validation R2:", study.best_value)

    # set hyperparameters
    learning_rate = study.best_params["learning_rate"]
    dropout_p = study.best_params["dropout_p"]
    l2 = study.best_params["l2"]

    # Instantiate the model
    model = DNN(
        input_dim=input_dim,
        output_dim=output_dim,
        hidden_dims=hidden_dims,
        use_dropout=use_dropout,
        dropout_p=dropout_p,
        use_batch_norm=use_batch_norm,
    )

    # Define loss function and optimizer
    loss_fn = nn.MSELoss()
    # Define optimizer
    if l2 > 0:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=l2)
    else:
        optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    # optimizer = torch.optim.SGD(model.parameters(), lr=1e-4, momentum=0.9)
    scheduler = ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=5, verbose=True)

    # parallelize model and send to machine
    model = nn.DataParallel(model)
    model.to(device)

    # Early stopping parameters
    patience = 20
    best_loss = float('inf')
    epochs_no_improve = 0

    # COMMENT IN IF YOU WANT TO TRACK MODEL TRAINING WITH TENSORBOARD
    # create summary writer directory name
    # writer_dir = Path(writer_path) / f"run{tune_split}"

    # # Initialize TensorBoard writer
    # print("\n", "... Initializing TensorBoard writer ...")
    # print(" > TensorBoard writer directory:", writer_dir, "\n")
    # writer = SummaryWriter(writer_dir)

    # print 
    print("training model for CV split " + str(tune_split))

    # start model training
    for epoch in tqdm(range(epochs)):
        
        # set model for backprop
        model.train()
        train_loss = 0.0
        
        # start batchs for training
        for i, (inputs, labels, ids) in enumerate(trainloader, 0):
            if torch.isnan(inputs).any():
                print("NaN detected in input")
                break 
            
            optimizer.zero_grad()
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, labels.squeeze())
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        # COMMENT IN IF YOU WANT TO TRACK MODEL TRAINING WITH TENSORBOARD
        # # record training loss
        # writer.add_scalar('Train Loss', train_loss / len(trainloader), epoch)
            
        # model in the validation set
        model.eval()
        val_loss = 0.0
        valid_preds = []
        valid_labels = []
        valid_ids = []
        valid_preds_prs = []
        
        with torch.no_grad():
            for inputs, labels, ids in validloader:
                outputs = model(inputs).squeeze()
                loss = loss_fn(outputs, labels.squeeze())
                val_loss += loss.item()
                valid_preds.append(outputs.cpu().numpy())
                valid_labels.append(labels.cpu().numpy())
                valid_ids.extend(ids.cpu().numpy())
                valid_preds_prs.extend(outputs.cpu().numpy().flatten())
            
        # Flatten lists of predictions and labels
        valid_preds = np.concatenate(valid_preds)
        valid_labels = np.concatenate(valid_labels)
        
        # performance metrics
        valid_n = len(valid_labels)
        valid_p = inputs.shape[1]
        valid_r2 = r2_score(valid_labels, valid_preds)
        valid_adj_r2 = adjusted_r2(valid_labels, valid_preds, valid_n, valid_p)

        # step the scheduler
        scheduler.step(valid_r2)
        
        # COMMENT IN IF YOU WANT TO TRACK MODEL TRAINING WITH TENSORBOARD
        # # write to tensorboard 
        # writer.add_scalar('Valid Loss', val_loss / len(validloader), epoch)
        # writer.add_scalar("Valid R2", valid_r2, epoch)
        # writer.add_scalar("Valid Adjusted-R2", valid_adj_r2, epoch)

        # Check for early stopping
        current_val_loss = val_loss / len(validloader)       
        if current_val_loss < best_loss:  # Compare with best_loss
            best_loss = current_val_loss  # Update best_loss
            epochs_no_improve = 0  # Reset patience counter
            best_model_state = copy.deepcopy(model.state_dict())  # Save model state
        else:
            epochs_no_improve += 1  # Increment patience counter
            if epochs_no_improve == patience:
                print("Early stopping!")
                break
                
    # load best model from training
    model.load_state_dict(best_model_state)

    # run in test set
    model.eval()
    test_loss = 0.0
    test_preds = []
    test_labels = []
    test_ids = []
    test_preds_prs = []
                            
    with torch.no_grad():
        for inputs, labels, ids in testloader:
            outputs = model(inputs).squeeze()
            loss = loss_fn(outputs, labels.squeeze())
            test_loss +=loss.item()
            
            # record results
            test_preds.append(outputs.cpu().numpy())
            test_labels.append(labels.cpu().numpy())
            test_ids.extend(ids.cpu().numpy())
            test_preds_prs.extend(outputs.cpu().numpy().flatten())
            
    # Flatten lists of predictions and labels
    test_preds = np.concatenate(test_preds)
    test_labels = np.concatenate(test_labels)

    # save raw pred probs
    test_pred_probs = list(zip(test_ids, test_preds_prs))
            
    # performance metrics
    test_n = len(test_labels)
    test_p = inputs.shape[1]
    test_r2 = r2_score(test_labels, test_preds)
    test_adj_r2 = adjusted_r2(test_labels, test_preds, test_n, test_p)

    # print test performance metrics
    print('Test adjusted R2 split'+str(tune_split), test_adj_r2)
    print('Test R2 split'+str(tune_split), test_r2)
    print('\n') 

    # set key for recording
    key = "split"+str(tune_split)

    # set dict for recording
    test_dict = {"ids":test_ids, "phenotype":test_labels, "pred_probs":test_preds_prs ,"adj_r2":test_adj_r2, "r2":test_r2}

    # save
    test_data[key] = test_dict

# print
print("| ---- Finished DNN training ---- |", "\n")

# --- print total cell time ---
cell_elapsed = time.time() - cell_start
print(f"| ---- Finished neural network in {cell_elapsed/60:.2f} minutes ---- |", "\n")

[I 2025-11-17 09:50:53,417] A new study created in memory with name: no-name-0cf43261-e300-4068-8077-1b9f827ea923


| ... CV split 1  ... |
 > Train set shape: (30000, 293) (30000,)
 > Validation set shape: (10000, 293) (10000,)
 > Test set shape: (10000, 293) (10000,)
 > Input dimension: 293
 > Hidden layers: [512, 128]
 > Use dropout: [True, False]
 > Use batch norm: [False, False]

 > Performing grid search ...

| ... Setting device ... |
 > Using MPS device: mps  ... 
... Optimizing hyper-parameters with Optuna ...


 26%|██▌       | 26/100 [01:49<05:10,  4.20s/it]
[I 2025-11-17 09:52:42,647] Trial 0 finished with value: 0.4817905583135855 and parameters: {'learning_rate': 0.0001, 'dropout_p': 0.0, 'l2': 1e-06}. Best is trial 0 with value: 0.4817905583135855.


Early stopping!

| ... Setting device ... |
 > Using MPS device: mps  ... 
... Optimizing hyper-parameters with Optuna ...


 45%|████▌     | 45/100 [03:14<03:57,  4.32s/it]
[I 2025-11-17 09:55:57,089] Trial 1 finished with value: 0.4881400411074812 and parameters: {'learning_rate': 1e-05, 'dropout_p': 0.25, 'l2': 1e-06}. Best is trial 1 with value: 0.4881400411074812.


...

 64%|██████▍   | 64/100 [05:20<03:00,  5.01s/it]

Epoch 00064: reducing learning rate of group 0 to 1.5625e-07.


 67%|██████▋   | 67/100 [05:40<02:47,  5.08s/it]

Early stopping!


Test adjusted R2 split5 0.48860306001833387
Test R2 split5 0.5035884889026851


| ---- Finished DNN training ---- | 

| ---- Finished neural network in 258.39 minutes ---- | 



In [12]:
# convert results to df
test_df = pd.DataFrame.from_dict(test_data).transpose()

### Check model performance & compare with published result

The cell below prints the mean and standard deviation of the variance-explained across the test splits.

In [13]:
# check mean r2 across test splits
print("Mean test R2:", test_df['r2'].mean())

# check sd r2 across test splits
print("SD test R2:", test_df['r2'].std())

Mean test R2: 0.49608824811446317
SD test R2: 0.005841093233767484


In the cells below we:
- load the pickle file containined the original (published) results for the same phenotype
- and print the mean and SD of the variance explained in the test set

These results should be *highly* consistent with the local run - with variation seen here < 0.001. This variation is expected, as software environments are slightly different between the local (this notebook) and the HPC (published) versions      

(see documentation: https://glaringlee.github.io/notes/randomness.html?)

In [14]:
# load published results for comparison
with open(pubfile, "rb") as f:
    published_results = pickle.load(f)

In [15]:
# check mean r2 across test splits
print("Mean test R2:", published_results['r2'].mean())

# check sd r2 across test splits
print("SD test R2:", published_results['r2'].std())

Mean test R2: 0.4958085038361748
SD test R2: 0.005164005806437506
